In [2]:
import numpy as np
pl = np.load('Unsup_labels/UCF_unsup_labels_original_V2.npy')
pl.shape

(779951,)

In [5]:

import numpy as np
s = np.load('C:/Users/jplabuser/Downloads/UCF_test_feature/UCF_test_feature/Normal_Videos_641_x264_i3d.npy')
s.shape

(225, 10, 2048)

In [12]:
nalist = np.load("list/nalist.npy")
print("nalist[-1,1] =", int(nalist[-1,1]))
nalist.shape


nalist[-1,1] = 779951


(1609, 2)

In [33]:
import numpy as np
nalist = np.load(r"../../C2FPL/list/nalist_i3d.npy")
pl  = np.load(r"Unsup_labels/UCF_unsup_labels_original_V2.npy")

assert (nalist[:,0] >= 0).all()
assert (nalist[:,1] > nalist[:,0]).all()
assert (nalist[1:,0] == nalist[:-1,1]).all()   # 구간이 딱 이어붙는지


means = []
for a,b in nalist[:50]:
    a=int(a); b=int(b)
    means.append(pl[a:b].mean())
print("first50 mean range:", min(means), max(means))

len_auth = nalist[:,1]-nalist[:,0]
print("len_auth median/mean/min/max:",
      np.median(len_auth), len_auth.mean(), len_auth.min(), len_auth.max())

# 각 비디오 길이 합이 pseudo 길이인지
assert int((nalist[:,1]-nalist[:,0]).sum()) == len(pl)


first50 mean range: 0.0 1.0
len_auth median/mean/min/max: 140.0 484.74269732753265 7 61032


In [29]:
import numpy as np
na = np.load(r"../../C2FPL/list/nalist.npy")
ny = np.load(r"../../C2FPL/list/nalist_i3d.npy")

print("first 10 lengths auth:", (na[:10,1]-na[:10,0]).astype(int))
print("first 10 lengths yours:", (ny[:10,1]-ny[:10,0]).astype(int))
print("same ratio:", ((na[:,1]-na[:,0])==(ny[:,1]-ny[:,0])).mean())


first 10 lengths auth: [ 34 104  58  25  62 161  47  57 147  76]
first 10 lengths yours: [ 171   55  232 1050   60  274   72  526   63   71]
same ratio: 0.0037290242386575512


In [30]:
import numpy as np, os

paths = [line.strip().split()[0] for line in open(r"../../C2FPL/list/ucf-i3d_train_fixed_local.list") if line.strip()]
for i in range(5):
    feat = np.load(paths[i], mmap_mode="r")
    print(i, os.path.basename(paths[i]), "T=", feat.shape[0])


0 Abuse001_x264_i3d.npy T= 171
1 Abuse002_x264_i3d.npy T= 55
2 Abuse003_x264_i3d.npy T= 232
3 Abuse004_x264_i3d.npy T= 1050
4 Abuse005_x264_i3d.npy T= 60


In [4]:
pl_2 = np.load('../../nayoung/Unsup_labels/UCF_hard_label.npy')
pl_2.shape

(779951,)

In [19]:
nalist_2 = np.load("../../C2FPL/list/nalist_i3d.npy")
print("nalist[-1,1] =", int(nalist_2[-1,1]))
#nalist_2.shape

nalist_XD = np.load("list/nalist_XD_test.npy")
print("nalist[-1,1] =", int(nalist_XD[-1,1]))
nalist_XD.shape


nalist[-1,1] = 779951
nalist[-1,1] = 145649


(800, 2)

In [8]:
import numpy as np

nal_auth = np.load(r"../../C2FPL/list/nalist.npy")
nal_yours = np.load(r"../../C2FPL/list/nalist_i3d.npy")

len_auth = nal_auth[:,1] - nal_auth[:,0]
len_yours = nal_yours[:,1] - nal_yours[:,0]

print("num videos:", len(nal_auth), len(nal_yours))
print("same lengths ratio:", (len_auth==len_yours).mean())
print("median auth/yours:", np.median(len_auth), np.median(len_yours))
print("max abs diff:", np.max(np.abs(len_auth - len_yours)))

# 첫 10개 차이 보기
for i in range(10):
    print(i, len_auth[i], len_yours[i], "diff", int(len_yours[i]-len_auth[i]))


num videos: 1609 1609
same lengths ratio: 0.0037290242386575512
median auth/yours: 140.0 140.0
max abs diff: 60992
0 34 171 diff 137
1 104 55 diff -49
2 58 232 diff 174
3 25 1050 diff 1025
4 62 60 diff -2
5 161 274 diff 113
6 47 72 diff 25
7 57 526 diff 469
8 147 63 diff -84
9 76 71 diff -5


In [9]:
import numpy as np, random

paths = [line.strip().split()[0] for line in open("../../C2FPL/list/ucf-i3d_train_fixed_local.list")]
nalist = np.load("../../C2FPL/list/nalist.npy")
total_T = int(nalist[-1,1])
con = np.memmap("../../C2FPL/concat_UCF.npy", dtype="float32", mode="r", shape=(total_T,10,2048))

for i in random.sample(range(len(paths)), 5):
    a,b = map(int, nalist[i])
    feat = np.load(paths[i])  # (T,10,2048)
    d1 = np.max(np.abs(con[a] - feat[0]))
    d2 = np.max(np.abs(con[b-1] - feat[-1]))
    print(i, "T:", feat.shape[0], "diff:", d1, d2)


1055 T: 135 diff: 5.4036703 4.6491394
1506 T: 64 diff: 7.4442024 6.7169404
479 T: 102 diff: 7.2009144 5.873343
120 T: 47 diff: 3.894911 5.759172
1119 T: 113 diff: 7.6965055 7.849602


In [17]:
import os

def vid_key(p):
    base = os.path.basename(p)
    # 예: Normal_XXX_x264.npy 같은 규칙이면 prefix만
    return base.split('.')[0]

auth_list = [line.split()[0].strip() for line in open(r"../../C2FPL/list/ucf-i3d.list", encoding="utf-8") if line.strip()]
your_list = [line.split()[0].strip() for line in open(r"../../C2FPL/list/ucf-i3d_train_fixed_local.list", encoding="utf-8") if line.strip()]

same = sum(vid_key(a)==vid_key(b) for a,b in zip(auth_list, your_list))
print("same order ratio:", same/len(auth_list))

# 틀린 인덱스 몇 개 보기
bad = [i for i,(a,b) in enumerate(zip(auth_list, your_list)) if vid_key(a)!=vid_key(b)]
print("num mismatches:", len(bad), "first mism:", bad[:10])
if bad:
    i = bad[0]
    print("auth:", auth_list[i])
    print("yours:", your_list[i])


same order ratio: 1.0
num mismatches: 0 first mism: []


내가 만든 pseudo 의 이상 비율

In [3]:
p = np.load('../../nayoung/Unsup_labels/UCF_hard_label.npy')  # np array
print("pos ratio:", p.mean(), "min/max:", p.min(), p.max())

pos ratio: 0.05977939639797885 min/max: 0 1


In [17]:
import numpy as np, os, random

na = np.load(r"../../C2FPL/list/nalist.npy")
len_auth = (na[:,1]-na[:,0]).astype(int)

paths = [line.strip().split()[0] for line in open(r"../../C2FPL/list/ucf-i3d_train_fixed_local.list") if line.strip()]

ratios=[]
for i in random.sample(range(len(paths)), 50):
    T = np.load(paths[i], mmap_mode="r").shape[0]
    ratios.append(T / max(1,len_auth[i]))
    
print("ratio median/min/max:", np.median(ratios), min(ratios), max(ratios))


ratio median/min/max: 0.8338755980861243 0.0006553938917289291 81.62068965517241


In [1]:
import numpy as np

paths = [line.strip().split()[0] for line in open("../../C2FPL/list/ucf-i3d_train_fixed_local.list") if line.strip()]
ny = np.load("../../C2FPL/list/nalist_i3d.npy")
ly = (ny[:,1]-ny[:,0]).astype(int)

bad = []
for i in range(len(paths)):
    T = np.load(paths[i], mmap_mode="r").shape[0]
    if T != ly[i]:
        bad.append((i, T, ly[i], paths[i]))

print("mismatch count:", len(bad))
print("first 10 mismatches:", bad[:10])

mismatch count: 0
first 10 mismatches: []


In [5]:
import numpy as np
na = np.load("../../C2FPL/list/nalist.npy")
ny = np.load("../../C2FPL/list/nalist_i3d.npy")
la = (na[:,1]-na[:,0]).astype(int)
ly = (ny[:,1]-ny[:,0]).astype(int)

# 각 길이의 등장 횟수
ua, ca = np.unique(la, return_counts=True)
unique_lengths = set(ua[ca==1])

idx = [i for i in range(len(ly)) if ly[i] in unique_lengths]
print("num uniquely-identifiable videos:", len(idx))

# 몇 개만 찍어서 "이 비디오는 저자 쪽에서 몇 번째였나" 확인
pos_in_auth = {L: int(np.where(la==L)[0][0]) for L in unique_lengths}
for i in idx[:20]:
    L = ly[i]
    print("your idx", i, "len", L, "auth idx", pos_in_auth[L])


num uniquely-identifiable videos: 285
your idx 3 len 1050 auth idx 802
your idx 7 len 526 auth idx 806
your idx 32 len 166 auth idx 831
your idx 36 len 884 auth idx 835
your idx 39 len 1780 auth idx 838
your idx 42 len 318 auth idx 841
your idx 47 len 304 auth idx 846
your idx 49 len 191 auth idx 848
your idx 53 len 400 auth idx 852
your idx 57 len 1011 auth idx 856
your idx 61 len 662 auth idx 860
your idx 63 len 153 auth idx 862
your idx 65 len 184 auth idx 864
your idx 70 len 406 auth idx 869
your idx 73 len 486 auth idx 872
your idx 76 len 485 auth idx 875
your idx 77 len 727 auth idx 876
your idx 78 len 387 auth idx 877
your idx 88 len 3942 auth idx 887
your idx 90 len 738 auth idx 889


In [6]:
import os

auth_list = [line.split()[0].strip() for line in open(r"../../C2FPL/list/ucf-i3d.list", encoding="utf-8") if line.strip()]
your_list = [line.split()[0].strip() for line in open(r"../../C2FPL/list/ucf-i3d_train_fixed_local.list", encoding="utf-8") if line.strip()]

for i in range(20):
    print(i, os.path.basename(auth_list[i]), " | ", os.path.basename(your_list[i]))


0 Abuse001_x264_i3d.npy  |  Abuse001_x264_i3d.npy
1 Abuse002_x264_i3d.npy  |  Abuse002_x264_i3d.npy
2 Abuse003_x264_i3d.npy  |  Abuse003_x264_i3d.npy
3 Abuse004_x264_i3d.npy  |  Abuse004_x264_i3d.npy
4 Abuse005_x264_i3d.npy  |  Abuse005_x264_i3d.npy
5 Abuse006_x264_i3d.npy  |  Abuse006_x264_i3d.npy
6 Abuse007_x264_i3d.npy  |  Abuse007_x264_i3d.npy
7 Abuse008_x264_i3d.npy  |  Abuse008_x264_i3d.npy
8 Abuse009_x264_i3d.npy  |  Abuse009_x264_i3d.npy
9 Abuse010_x264_i3d.npy  |  Abuse010_x264_i3d.npy
10 Abuse011_x264_i3d.npy  |  Abuse011_x264_i3d.npy
11 Abuse012_x264_i3d.npy  |  Abuse012_x264_i3d.npy
12 Abuse013_x264_i3d.npy  |  Abuse013_x264_i3d.npy
13 Abuse014_x264_i3d.npy  |  Abuse014_x264_i3d.npy
14 Abuse015_x264_i3d.npy  |  Abuse015_x264_i3d.npy
15 Abuse016_x264_i3d.npy  |  Abuse016_x264_i3d.npy
16 Abuse017_x264_i3d.npy  |  Abuse017_x264_i3d.npy
17 Abuse018_x264_i3d.npy  |  Abuse018_x264_i3d.npy
18 Abuse019_x264_i3d.npy  |  Abuse019_x264_i3d.npy
19 Abuse020_x264_i3d.npy  |  Abuse020_x26

In [8]:
import os, numpy as np
na = np.load("../../C2FPL/list/nalist.npy"); ny = np.load("../../C2FPL/list/nalist_i3d.npy")
la = (na[:,1]-na[:,0]).astype(int)
ly = (ny[:,1]-ny[:,0]).astype(int)

auth_list = [line.split()[0].strip() for line in open(r"../../C2FPL/list/ucf-i3d.list", encoding="utf-8") if line.strip()]
your_list = [line.split()[0].strip() for line in open(r"../../C2FPL/list/ucf-i3d_train_fixed_local.list", encoding="utf-8") if line.strip()]

# 고유 길이 하나 선택
ua, ca = np.unique(la, return_counts=True)
unique_lengths = set(ua[ca==1])

for i in range(len(ly)):
    L = ly[i]
    if L in unique_lengths:
        j = int(np.where(la==L)[0][0])  # auth idx
        print("unique len", L)
        print("your idx", i, os.path.basename(your_list[i]))
        print("auth idx", j, os.path.basename(auth_list[j]))
        break


unique len 1050
your idx 3 Abuse004_x264_i3d.npy
auth idx 802 Vandalism043_x264_i3d.npy
